<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distributions

I will first inspect the distributions of the main numeric signals used for content refresh prioritization. I will look at search volume, 90-day impressions, word count, and other available numeric fields. Heavy-tailed fields will be checked using medians and upper percentiles rather than relying only on averages.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Load the repository and inspect distributions

import os
import pandas as pd
import numpy as np

REPO_DIR = "/content/AIML"

# Clone the repository only if it is not already present
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/vaishnavikabbe/AIML.git /content/AIML

os.chdir(REPO_DIR)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Check that the AIML repository contains the starter dataset."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Main numeric fields
fields = [
    "search_volume",
    "impressions_90d",
    "word_count"
]

print("\nDistribution summary:")

for col in fields:
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")

        print(f"\n{col}")
        print("Median:", round(s.median(), 2))
        print("75th percentile:", round(s.quantile(0.75), 2))
        print("95th percentile:", round(s.quantile(0.95), 2))
        print("Missing:", int(s.isna().sum()))

Dataset loaded successfully.
Rows: 30000
Columns: 44

Distribution summary:

search_volume
Median: 10.0
75th percentile: 20.0
95th percentile: 390.0
Missing: 2468

impressions_90d
Median: 731.0
75th percentile: 3615.25
95th percentile: 22996.5
Missing: 0

word_count
Median: 2877.0
75th percentile: 3666.0
95th percentile: 6173.0
Missing: 7699


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal tests

I will test three signals that may be useful for content refresh prioritization: search volume, 90-day impressions, and word count. For each signal, I will compare its values across the observed trend directions. The verdicts will be based on the measured differences rather than on assumptions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Test three signals against the observed trend direction

if "trend_direction" not in df.columns:
    raise KeyError("trend_direction column not found in the dataset.")

print("Trend distribution:")
print(df["trend_direction"].value_counts(dropna=False))

signals = [
    "search_volume",
    "impressions_90d",
    "word_count"
]

for signal in signals:
    if signal not in df.columns:
        print(f"\n{signal}: column not found - skipped")
        continue

    print(f"\n{'='*50}")
    print(f"Signal: {signal}")

    summary = (
        df.groupby("trend_direction")[signal]
        .median()
        .sort_values()
    )

    print("Median by trend direction:")
    print(summary)

    # Simple spread between highest and lowest trend-group median
    if len(summary) >= 2:
        difference = summary.iloc[-1] - summary.iloc[0]
        print("Median difference:", round(difference, 2))

        if difference == 0:
            verdict = "FALSE"
        else:
            verdict = "MIXED"

        print("Verdict:", verdict)
    else:
        print("Verdict: MIXED")

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Signal: search_volume
Median by trend direction:
trend_direction
down      10.0
flat      10.0
new       10.0
stable    10.0
up        10.0
Name: search_volume, dtype: float64
Median difference: 0.0
Verdict: FALSE

Signal: impressions_90d
Median by trend direction:
trend_direction
new          3.0
flat         4.0
up         587.0
down       961.0
stable    1944.5
Name: impressions_90d, dtype: float64
Median difference: 1941.5
Verdict: MIXED

Signal: word_count
Median by trend direction:
trend_direction
new       2239.0
flat      2698.5
up        2847.5
down      2909.0
stable    2912.5
Name: word_count, dtype: float64
Median difference: 673.5
Verdict: MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test

I will test whether search volume is strongly associated with 90-day impressions, because a common search-performance assumption is that pages with higher search demand should also receive proportionally higher impressions. The test will use the observed correlation in the anonymized dataset. A weak relationship will be treated as evidence against using search volume alone as a prioritization rule.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Flag-linked signal test

required = ["search_volume", "impressions_90d"]

for col in required:
    if col not in df.columns:
        raise KeyError(f"Required column not found: {col}")

test_df = df[required].copy()

test_df["search_volume"] = pd.to_numeric(
    test_df["search_volume"], errors="coerce"
)

test_df["impressions_90d"] = pd.to_numeric(
    test_df["impressions_90d"], errors="coerce"
)

test_df = test_df.dropna()

correlation = test_df["search_volume"].corr(
    test_df["impressions_90d"]
)

print("Rows used:", len(test_df))
print(
    "Correlation between search_volume and impressions_90d:",
    round(correlation, 3)
)

if abs(correlation) >= 0.5:
    verdict = "CONFIRMED"
elif abs(correlation) >= 0.2:
    verdict = "MIXED"
else:
    verdict = "FALSE"

print("Flag-linked test verdict:", verdict)

Rows used: 27532
Correlation between search_volume and impressions_90d: 0.001
Flag-linked test verdict: FALSE


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### What this means in practice

The observed data suggests that content teams should not rely on a single signal such as search volume when deciding which pages to review. The signals show different patterns across the observed trend groups, so a combination of measured signals may provide better decision-support. These findings are directional and should be used to prioritize further review rather than as causal proof.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Print the practical takeaway

print("Practical takeaway:")
print(
    "Search volume alone should not be treated as a sufficient "
    "content-refresh prioritization rule."
)

print(
    "A combination of observed search and content signals "
    "can be investigated for better decision-support."
)

print(
    "Conclusion type: observed, measured, directional, decision-support"
)

Practical takeaway:
Search volume alone should not be treated as a sufficient content-refresh prioritization rule.
A combination of observed search and content signals can be investigated for better decision-support.
Conclusion type: observed, measured, directional, decision-support


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.